<a href="https://colab.research.google.com/github/Zeldano118/QPon_NLP_PBA/blob/main/assignments/week%204/1_week4_tfidf_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4: TF-IDF Text Summarization
Based on Colab (Danantara + Manchester). Tugas 1A, 1B, 1C.

- **1A:** Danantara article (Indonesian)
- **1B:** Manchester article (English)
- **1C:** Berita banjir article (Indonesian) —> replaced with own news article

In [ ]:
!pip install Sastrawi -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import matplotlib.pyplot as plt
import pandas as pd
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 9.3 MB/s eta 0:00:00


---
## Tugas 1B: Manchester (English)

In [ ]:
sentence_en = """
Manchester City makes history by winning Club World Cup

Manchester City capped off its incredible year with yet another trophy, dismantling Fluminense 4-0 to win the Club World Cup on Friday.

Having already won the Premier League, Champions League, FA Cup and Super Cup, Pep Guardiola's side now boasts five trophies this calendar year, becoming the first English club to ever hold all those titles simultaneously.

The final piece of the jigsaw came on a highly charged night in Saudi Arabia as Manchester City outclassed its Brazilian opponents.

We've shown over the past 12 months that we are the best team in the world, Guardiola said after the match. It means a lot to me and to this club.

Julian Alvarez opened the scoring in the first half before Phil Foden doubled the lead with a sublime strike. Alvarez completed his brace in the second half, and substitute Nathan Ake wrapped things up in stoppage time.

Fluminense, the reigning Copa Libertadores champion, struggled to contain City's relentless pressing and quick passing combinations throughout the match.
"""

In [ ]:
sent_token_en = sent_tokenize(sentence_en)
print(f'{len(sent_token_en)} sentences\n')
for i, s in enumerate(sent_token_en):
    print(f'{i+1}. {s}')

8 sentences

1. 
Manchester City makes history by winning Club World Cup

Manchester City capped off its incredible year with yet another trophy, dismantling Fluminense 4-0 to win the Club World Cup on Friday.
2. Having already won the Premier League, Champions League, FA Cup and Super Cup, Pep Guardiola's side now boasts five trophies this calendar year, becoming the first English club to ever hold all those titles simultaneously.
3. The final piece of the jigsaw came on a highly charged night in Saudi Arabia as Manchester City outclassed its Brazilian opponents.
4. We've shown over the past 12 months that we are the best team in the world, Guardiola said after the match.
5. It means a lot to me and to this club.
6. Julian Alvarez opened the scoring in the first half before Phil Foden doubled the lead with a sublime strike.
7. Alvarez completed his brace in the second half, and substitute Nathan Ake wrapped things up in stoppage time.
8. Fluminense, the reigning Copa Libertadores cham

In [ ]:
vectorizer_en = TfidfVectorizer(stop_words='english')
features_en = vectorizer_en.fit_transform(sent_token_en)

# sentence scores
scores_en = features_en.sum(axis=1).A1
print('Sentence scores:')
for i, (s, sc) in enumerate(zip(sent_token_en, scores_en)):
    print(f'  [{sc:.2f}] {s[:80]}...' if len(s) > 80 else f'  [{sc:.2f}] {s}')

threshold_en = sum(scores_en) / len(scores_en)
print(f'\nThreshold: {threshold_en:.2f}')

Sentence scores:
  [3.88] 
Manchester City makes history by winning Club World Cup

Manchester City capped...
  [4.21] Having already won the Premier League, Champions League, FA Cup and Super Cup, P...
  [3.73] The final piece of the jigsaw came on a highly charged night in Saudi Arabia as ...
  [3.31] We've shown over the past 12 months that we are the best team in the world, Guar...
  [1.71] It means a lot to me and to this club.
  [3.31] Julian Alvarez opened the scoring in the first half before Phil Foden doubled th...
  [3.46] Alvarez completed his brace in the second half, and substitute Nathan Ake wrappe...
  [3.73] Fluminense, the reigning Copa Libertadores champion, struggled to contain City's...

Threshold: 3.42


In [ ]:
print('Summary (English):\n')
summary_en = ''
for i, sc in enumerate(scores_en):
    if sc >= threshold_en:
        summary_en += ' ' + sent_token_en[i]
        print(f'- {sent_token_en[i]}')
print(f'\nSummary length: {len(summary_en.split())} words (from {len(sentence_en.split())} original)')

Summary (English):

- 
Manchester City makes history by winning Club World Cup

Manchester City capped off its incredible year with yet another trophy, dismantling Fluminense 4-0 to win the Club World Cup on Friday.
- Having already won the Premier League, Champions League, FA Cup and Super Cup, Pep Guardiola's side now boasts five trophies this calendar year, becoming the first English club to ever hold all those titles simultaneously.
- The final piece of the jigsaw came on a highly charged night in Saudi Arabia as Manchester City outclassed its Brazilian opponents.
- Alvarez completed his brace in the second half, and substitute Nathan Ake wrapped things up in stoppage time.
- Fluminense, the reigning Copa Libertadores champion, struggled to contain City's relentless pressing and quick passing combinations throughout the match.

Summary length: 125 words (from 175 original)


---
## Tugas 1A: Danantara (Indonesian)

In [ ]:
sentence_id = """Jakarta: Badan Pengelola Investasi Daya Anagata Nusantara (BPI Danantara) siap mengawal realisasi investasi yang telah disepakati dengan Qatar. Kesepakatan antara Indonesia dan Qatar merupakan buah dari kunjungan resmi Presiden Prabowo Subianto ke Doha.

Pemerintah Republik Indonesia dan Pemerintah Qatar menggelar diskusi untuk menyepakati kemitraan strategis (co-partnership) dalam pengelolaan dana investasi untuk Indonesia yang akan berfokus di berbagai sektor pembangunan.

Salah satu hasil utama dari kunjungan tersebut adalah untuk membentuk dana investasi bersama senilai USD4 miliar atau setara dengan Rp 65 triliun. Danantara akan menjadi mitra utama dalam pengelolaan investasi tersebut, memastikan bahwa dana yang masuk dimanfaatkan secara strategis dan transparan.

Rosan menambahkan bahwa investasi ini akan difokuskan pada sektor-sektor prioritas seperti energi hijau, infrastruktur digital, ketahanan pangan, dan pariwisata berkelanjutan."""

In [ ]:
sent_token_id = sent_tokenize(sentence_id)
print(f'{len(sent_token_id)} sentences\n')
for i, s in enumerate(sent_token_id):
    print(f'{i+1}. {s}')

6 sentences

1. Jakarta: Badan Pengelola Investasi Daya Anagata Nusantara (BPI Danantara) siap mengawal realisasi investasi yang telah disepakati dengan Qatar.
2. Kesepakatan antara Indonesia dan Qatar merupakan buah dari kunjungan resmi Presiden Prabowo Subianto ke Doha.
3. Pemerintah Republik Indonesia dan Pemerintah Qatar menggelar diskusi untuk menyepakati kemitraan strategis (co-partnership) dalam pengelolaan dana investasi untuk Indonesia yang akan berfokus di berbagai sektor pembangunan.
4. Salah satu hasil utama dari kunjungan tersebut adalah untuk membentuk dana investasi bersama senilai USD4 miliar atau setara dengan Rp 65 triliun.
5. Danantara akan menjadi mitra utama dalam pengelolaan investasi tersebut, memastikan bahwa dana yang masuk dimanfaatkan secara strategis dan transparan.
6. Rosan menambahkan bahwa investasi ini akan difokuskan pada sektor-sektor prioritas seperti energi hijau, infrastruktur digital, ketahanan pangan, dan pariwisata berkelanjutan.


In [ ]:
# Indonesian stopwords via Sastrawi
factory = StopWordRemoverFactory()
stop_id = factory.get_stop_words()

vectorizer_id = TfidfVectorizer(stop_words=stop_id)
features_id = vectorizer_id.fit_transform(sent_token_id)

scores_id = features_id.sum(axis=1).A1
print('Sentence scores:')
for i, (s, sc) in enumerate(zip(sent_token_id, scores_id)):
    print(f'  [{sc:.2f}] {s[:80]}...' if len(s) > 80 else f'  [{sc:.2f}] {s}')

threshold_id = sum(scores_id) / len(scores_id)
print(f'\nThreshold: {threshold_id:.2f}')

Sentence scores:
  [3.73] Jakarta: Badan Pengelola Investasi Daya Anagata Nusantara (BPI Danantara) siap m...
  [3.30] Kesepakatan antara Indonesia dan Qatar merupakan buah dari kunjungan resmi Presi...
  [4.03] Pemerintah Republik Indonesia dan Pemerintah Qatar menggelar diskusi untuk menye...
  [4.08] Salah satu hasil utama dari kunjungan tersebut adalah untuk membentuk dana inves...
  [3.56] Danantara akan menjadi mitra utama dalam pengelolaan investasi tersebut, memasti...
  [3.66] Rosan menambahkan bahwa investasi ini akan difokuskan pada sektor-sektor priorit...

Threshold: 3.72


In [ ]:
print('Summary (Danantara):\n')
for i, sc in enumerate(scores_id):
    if sc >= threshold_id:
        print(f'- {sent_token_id[i]}')

Summary (Danantara):

- Jakarta: Badan Pengelola Investasi Daya Anagata Nusantara (BPI Danantara) siap mengawal realisasi investasi yang telah disepakati dengan Qatar.
- Pemerintah Republik Indonesia dan Pemerintah Qatar menggelar diskusi untuk menyepakati kemitraan strategis (co-partnership) dalam pengelolaan dana investasi untuk Indonesia yang akan berfokus di berbagai sektor pembangunan.
- Salah satu hasil utama dari kunjungan tersebut adalah untuk membentuk dana investasi bersama senilai USD4 miliar atau setara dengan Rp 65 triliun.


---
## Tugas 1C: Berita Banjir (Own News Article)
Replacing with a flood news article from my dataset.

In [ ]:
# Berita: Banjir di Katimbang Makassar Dipicu Penyempitan Aliran Sungai Biring Je'ne
# Source: Detik (https://www.detik.com/sulsel/makassar/d-8301766/)
sentence_news = """Banjir yang melanda kawasan Katimbang, Kecamatan Biringkanaya, Kota Makassar, diduga kuat dipicu oleh penyempitan aliran Sungai Biring Je'ne. Kondisi ini diperparah oleh curah hujan tinggi yang mengguyur Makassar dalam beberapa hari terakhir.

Kepala Pelaksana BPBD Kota Makassar mengatakan penyempitan sungai terjadi akibat pembangunan permukiman warga di bantaran sungai yang mempersempit aliran air. Saat hujan deras, air tidak mampu mengalir dengan baik sehingga meluap ke permukiman.

Setidaknya 94 warga harus mengungsi dari rumah mereka akibat ketinggian air yang mencapai satu meter. Warga dievakuasi ke masjid dan balai kelurahan setempat.

Pemerintah Kota Makassar berencana melakukan normalisasi Sungai Biring Je'ne untuk mencegah banjir berulang. Selain itu, BPBD juga mengerahkan pompa air untuk menyedot genangan di permukiman warga.

Banjir di Katimbang merupakan masalah tahunan yang kerap terjadi saat musim hujan. Warga berharap pemerintah segera menuntaskan proyek normalisasi sungai agar banjir tidak terulang setiap tahun."""

In [ ]:
sent_token_news = sent_tokenize(sentence_news)
print(f'{len(sent_token_news)} sentences\n')
for i, s in enumerate(sent_token_news):
    print(f'{i+1}. {s}')

10 sentences

1. Banjir yang melanda kawasan Katimbang, Kecamatan Biringkanaya, Kota Makassar, diduga kuat dipicu oleh penyempitan aliran Sungai Biring Je'ne.
2. Kondisi ini diperparah oleh curah hujan tinggi yang mengguyur Makassar dalam beberapa hari terakhir.
3. Kepala Pelaksana BPBD Kota Makassar mengatakan penyempitan sungai terjadi akibat pembangunan permukiman warga di bantaran sungai yang mempersempit aliran air.
4. Saat hujan deras, air tidak mampu mengalir dengan baik sehingga meluap ke permukiman.
5. Setidaknya 94 warga harus mengungsi dari rumah mereka akibat ketinggian air yang mencapai satu meter.
6. Warga dievakuasi ke masjid dan balai kelurahan setempat.
7. Pemerintah Kota Makassar berencana melakukan normalisasi Sungai Biring Je'ne untuk mencegah banjir berulang.
8. Selain itu, BPBD juga mengerahkan pompa air untuk menyedot genangan di permukiman warga.
9. Banjir di Katimbang merupakan masalah tahunan yang kerap terjadi saat musim hujan.
10. Warga berharap pemerintah s

In [ ]:
vectorizer_news = TfidfVectorizer(stop_words=stop_id)
features_news = vectorizer_news.fit_transform(sent_token_news)

scores_news = features_news.sum(axis=1).A1
print('Sentence scores:')
for i, (s, sc) in enumerate(zip(sent_token_news, scores_news)):
    print(f'  [{sc:.2f}] {s[:80]}...' if len(s) > 80 else f'  [{sc:.2f}] {s}')

threshold_news = sum(scores_news) / len(scores_news)
print(f'\nThreshold: {threshold_news:.2f}')

Sentence scores:
  [4.08] Banjir yang melanda kawasan Katimbang, Kecamatan Biringkanaya, Kota Makassar, di...
  [3.14] Kondisi ini diperparah oleh curah hujan tinggi yang mengguyur Makassar dalam beb...
  [4.05] Kepala Pelaksana BPBD Kota Makassar mengatakan penyempitan sungai terjadi akibat...
  [2.79] Saat hujan deras, air tidak mampu mengalir dengan baik sehingga meluap ke permuk...
  [3.12] Setidaknya 94 warga harus mengungsi dari rumah mereka akibat ketinggian air yang...
  [2.42] Warga dievakuasi ke masjid dan balai kelurahan setempat.
  [3.57] Pemerintah Kota Makassar berencana melakukan normalisasi Sungai Biring Je'ne unt...
  [2.78] Selain itu, BPBD juga mengerahkan pompa air untuk menyedot genangan di permukima...
  [2.97] Banjir di Katimbang merupakan masalah tahunan yang kerap terjadi saat musim huja...
  [3.27] Warga berharap pemerintah segera menuntaskan proyek normalisasi sungai agar banj...

Threshold: 3.22


In [ ]:
print('Summary (Berita Banjir):\n')
for i, sc in enumerate(scores_news):
    if sc >= threshold_news:
        print(f'- {sent_token_news[i]}')

Summary (Berita Banjir):

- Banjir yang melanda kawasan Katimbang, Kecamatan Biringkanaya, Kota Makassar, diduga kuat dipicu oleh penyempitan aliran Sungai Biring Je'ne.
- Kepala Pelaksana BPBD Kota Makassar mengatakan penyempitan sungai terjadi akibat pembangunan permukiman warga di bantaran sungai yang mempersempit aliran air.
- Pemerintah Kota Makassar berencana melakukan normalisasi Sungai Biring Je'ne untuk mencegah banjir berulang.
- Warga berharap pemerintah segera menuntaskan proyek normalisasi sungai agar banjir tidak terulang setiap tahun.
